# Computer Exercise 13.8 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.)
> **단원**: 13.8 Minimization — *The Levenberg–Marquardt Method*
> **풀이 일자**: 2026-06-27 · **언어**: 본문 한국어 / 그래프 라벨 영문


## 1. 문제 (원문)

> **2.** The Gauss–Newton step can fail when $J^\top J$ is (nearly) singular or when the initial
> guess is far from the solution. The **Levenberg–Marquardt (LM)** method damps the normal
> equations,
> $$ (J^\top J+\lambda\,D)\,\boldsymbol\delta=-J^\top\mathbf r, $$
> where $\lambda\ge0$ is adapted at each step and $D$ is a positive diagonal (often $D=\operatorname{diag}(J^\top J)$).
> Implement LM with an adaptive $\lambda$ controlled by the **gain ratio**
> $\rho=(f_k-f_{k+1})/(\text{predicted reduction})$. Fit a **Gaussian peak**
> $\varphi(t;A,\mu,\sigma)=A\,e^{-(t-\mu)^2/2\sigma^2}$ from a **poor initial guess**, and compare
> LM against plain Gauss–Newton and steepest descent. Track how $\lambda$ evolves.

### 한국어 풀이용 정리
- 모형: 가우시안 봉우리 $\varphi(t;A,\mu,\sigma)=A\,e^{-(t-\mu)^2/2\sigma^2}$, 파라미터 $\mathbf x=(A,\mu,\sigma)$.
- **감쇠 정규방정식** $(J^\top J+\lambda D)\boldsymbol\delta=-J^\top\mathbf r$ 를 푼다.
- **이득비** $\rho$ 로 $\lambda$ 를 자동 조절(성공→감소, 실패→증가).
- **나쁜 초기값**에서 LM vs Gauss–Newton vs 최급강하 비교, $\lambda$ 궤적 관찰.


## 2. 수학적 배경

### 2.1 감쇠(damping)의 의미
LM 스텝은 두 극단을 보간한다:
$$
(J^\top J+\lambda D)\,\boldsymbol\delta=-J^\top\mathbf r .
$$
* $\lambda\to0$ : $\boldsymbol\delta\to\boldsymbol\delta_{\mathrm{GN}}$ — **Gauss–Newton** 스텝(빠르지만 위험).
* $\lambda\to\infty$ : $\boldsymbol\delta\to-\tfrac1\lambda D^{-1}J^\top\mathbf r$ — **최급강하** 방향의 짧은 스텝(느리지만 안전).

$\lambda D$ 항은 $J^\top J+\lambda D\succ0$ 를 보장하므로, $J^\top J$ 가 특이해도 스텝이 항상 정의된다. 이는 신뢰영역(trust-region) 부분문제 $\min\;\tfrac12\lVert J\boldsymbol\delta+\mathbf r\rVert^2$ s.t. $\lVert D^{1/2}\boldsymbol\delta\rVert\le\Delta$ 의 라그랑주 형태와 동치다.

### 2.2 이득비에 의한 $\lambda$ 갱신
$$
\rho=\frac{f(\mathbf x_k)-f(\mathbf x_k+\boldsymbol\delta)}{-(\,\mathbf g^\top\boldsymbol\delta+\tfrac12\boldsymbol\delta^\top J^\top J\,\boldsymbol\delta\,)} .
$$
$\rho>0$ 이면 스텝 수락 후 $\lambda\!\leftarrow\!\lambda/3$ (GN 쪽), $\rho\le0$ 이면 기각 후 $\lambda\!\leftarrow\!3\lambda$ (최급강하 쪽).

### 2.3 가우시안 모형의 야코비안
$E_i=e^{-(t_i-\mu)^2/2\sigma^2}$ 라 하면 $r_i=A E_i-y_i$ 이고
$$
\frac{\partial r_i}{\partial A}=E_i,\quad
\frac{\partial r_i}{\partial \mu}=A E_i\frac{t_i-\mu}{\sigma^2},\quad
\frac{\partial r_i}{\partial \sigma}=A E_i\frac{(t_i-\mu)^2}{\sigma^3}.
$$


## 3. 풀이 흐름

1. **데이터 생성**: 참값 $(A,\mu,\sigma)=(3.0,\,2.0,\,0.7)$ 의 가우시안에 작은 잡음.
2. **모형/잔차/야코비안** 구현.
3. **LM 루프**: $D=\operatorname{diag}(J^\top J)$ 로 감쇠 정규방정식을 풀고, 이득비 $\rho$ 로 스텝 수락·$\lambda$ 갱신.
4. **Gauss–Newton 루프**(감쇠 없음)와 **최급강하**(백트래킹)를 같은 나쁜 초기값에서 실행.
5. **반복 표**: $k,A,\mu,\sigma,\lVert\mathbf r\rVert,\lVert\text{grad}\rVert,\lambda$.
6. **시각화**: (a) 세 방법의 cost 수렴, (b) LM 의 $\lambda$ 궤적, (c) 데이터·적합 곡선.
7. **해석**: 왜 LM 은 수렴하고 GN 은 폭주/진동하는가.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(3)

A_true, mu_true, s_true = 3.0, 2.0, 0.7
t = np.linspace(-1.0, 5.0, 40)
y_clean = A_true * np.exp(-(t - mu_true) ** 2 / (2 * s_true ** 2))
y = y_clean + 0.02 * rng.standard_normal(t.size)

def model(x, tt):
    A, mu, s = x
    return A * np.exp(-(tt - mu) ** 2 / (2 * s ** 2))

def residual(x):
    return model(x, t) - y

def jac(x):
    A, mu, s = x
    E = np.exp(-(t - mu) ** 2 / (2 * s ** 2))
    J = np.empty((t.size, 3))
    J[:, 0] = E
    J[:, 1] = A * E * (t - mu) / s ** 2
    J[:, 2] = A * E * (t - mu) ** 2 / s ** 3
    return J

x0_bad = np.array([0.6, 4.2, 1.8])
print("true (A,mu,s) =", (A_true, mu_true, s_true), "  bad x0 =", x0_bad)


true (A,mu,s) = (3.0, 2.0, 0.7)   bad x0 = [0.6 4.2 1.8]


In [2]:
def levenberg_marquardt(x0, lam0=1e-2, tol=1e-9, maxit=300):
    x = np.array(x0, float); lam = lam0
    r = residual(x); cost = 0.5 * r @ r
    hist = []
    for k in range(maxit):
        J = jac(x); g = J.T @ r; A = J.T @ J
        hist.append((k, x[0], x[1], x[2], np.sqrt(r @ r), np.linalg.norm(g), lam))
        if np.linalg.norm(g) < tol:
            break
        D = np.diag(np.maximum(np.diag(A), 1e-12))
        d = np.linalg.solve(A + lam * D, -g)
        x_new = x + d
        r_new = residual(x_new); cost_new = 0.5 * r_new @ r_new
        pred = -(g @ d + 0.5 * d @ A @ d)
        rho = (cost - cost_new) / pred if pred > 0 else -1.0
        if rho > 0:
            x, r, cost = x_new, r_new, cost_new
            lam = max(lam / 3.0, 1e-12)
        else:
            lam = lam * 3.0
    cols = ["k", "A", "mu", "s", "||r||", "||grad||", "lambda"]
    return x, pd.DataFrame(hist, columns=cols)

x_lm, hist_lm = levenberg_marquardt(x0_bad)
x_lm = x_lm.copy(); x_lm[2] = abs(x_lm[2])   # sigma는 제곱으로만 등장 -> 부호는 임의(대칭)
print("LM ->", x_lm, "  iters =", len(hist_lm) - 1)
x_lm


LM -> [3.00636771 2.00252545 0.69970497]   iters = 30


array([3.00636771, 2.00252545, 0.69970497])

In [3]:
# 비교용: 순수 Gauss-Newton, 최급강하(backtracking)
def gauss_newton(x0, tol=1e-9, maxit=300):
    x = np.array(x0, float); hist = []
    for k in range(maxit):
        r = residual(x); J = jac(x); g = J.T @ r
        hist.append((k, np.linalg.norm(g), 0.5 * r @ r))
        if np.linalg.norm(g) < tol:
            break
        try:
            d = np.linalg.solve(J.T @ J, -g)
        except np.linalg.LinAlgError:
            break
        x = x + d
        if not np.all(np.isfinite(x)) or np.linalg.norm(x) > 1e6:
            hist.append((k + 1, np.inf, np.inf))
            break
    return x, pd.DataFrame(hist, columns=["k", "||grad||", "cost"])

def steepest_descent(x0, tol=1e-9, maxit=4000):
    x = np.array(x0, float); hist = []
    for k in range(maxit):
        r = residual(x); J = jac(x); g = J.T @ r
        cost = 0.5 * r @ r
        hist.append((k, np.linalg.norm(g), cost))
        if np.linalg.norm(g) < tol:
            break
        step = 1.0
        for _ in range(50):
            xn = x - step * g
            if 0.5 * residual(xn) @ residual(xn) < cost - 1e-4 * step * (g @ g):
                break
            step *= 0.5
        x = x - step * g
    return x, pd.DataFrame(hist, columns=["k", "||grad||", "cost"])

with np.errstate(all="ignore"):
    x_gn, hist_gn = gauss_newton(x0_bad)
    x_sd, hist_sd = steepest_descent(x0_bad)
summary = pd.DataFrame({
    "method": ["Levenberg-Marquardt", "Gauss-Newton (pure)", "Steepest descent"],
    "iters": [len(hist_lm) - 1, len(hist_gn) - 1, len(hist_sd) - 1],
    "final cost": [hist_lm["||r||"].iloc[-1] ** 2 / 2,
                   hist_gn["cost"].iloc[-1], hist_sd["cost"].iloc[-1]],
    "converged": [hist_lm["||grad||"].iloc[-1] < 1e-6,
                  np.isfinite(hist_gn["cost"].iloc[-1]) and hist_gn["||grad||"].iloc[-1] < 1e-6,
                  hist_sd["||grad||"].iloc[-1] < 1e-6],
})
summary


,method,iters,final cost,converged
0,Levenberg-Marquardt,30,0.010412,True
1,Gauss-Newton (pure),3,inf,False
2,Steepest descent,3999,0.010412,True


In [4]:
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
pd.concat([hist_lm.head(8), hist_lm.tail(4)])


,k,A,mu,s,||r||,||grad||,lambda
0,0,6.000000e-01,4.200000e+00,1.800000e+00,7.649560e+00,1.060569e+01,1.000000e-02
1,1,6.000000e-01,4.200000e+00,1.800000e+00,7.649560e+00,1.060569e+01,3.000000e-02
2,2,6.000000e-01,4.200000e+00,1.800000e+00,7.649560e+00,1.060569e+01,9.000000e-02
3,3,6.000000e-01,4.200000e+00,1.800000e+00,7.649560e+00,1.060569e+01,2.700000e-01
4,4,3.261065e-01,7.555802e-02,5.233292e+00,7.457988e+00,2.121931e+01,9.000000e-02
5,5,8.049883e-01,4.055061e+01,-4.619493e+01,6.852513e+00,8.075130e+00,3.000000e-02
6,6,8.874992e-01,2.960833e+01,-5.952263e+01,6.617591e+00,2.080827e+00,1.000000e-02
7,7,8.735532e-01,1.038007e+01,-4.266106e+01,6.605342e+00,4.096653e-02,3.333333e-03
27,27,3.006278e+00,2.002525e+00,-6.997261e-01,1.443069e-01,8.691387e-04,3.333333e-03
28,28,3.006367e+00,2.002525e+00,-6.997052e-01,1.443067e-01,1.288696e-05,1.111111e-03


In [5]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))

ax[0].semilogy(hist_lm["k"], hist_lm["||r||"] ** 2 / 2, "o-", ms=4, color="#16a085", label="Levenberg-Marquardt")
ax[0].semilogy(hist_gn["k"], hist_gn["cost"], "s--", ms=4, color="#c0392b", label="Gauss-Newton")
ax[0].semilogy(hist_sd["k"], hist_sd["cost"], "-", lw=1.2, color="#7f8c8d", label="Steepest descent")
ax[0].set_xlabel("iteration k"); ax[0].set_ylabel("cost f(x)")
ax[0].set_title("Convergence from a poor start"); ax[0].legend(fontsize=8)

ax[1].semilogy(hist_lm["k"], hist_lm["lambda"], "o-", ms=4, color="#16a085")
ax[1].set_xlabel("iteration k"); ax[1].set_ylabel(r"damping $\lambda$")
ax[1].set_title("LM damping parameter trajectory")

tt = np.linspace(-1, 5, 250)
ax[2].plot(t, y, "o", ms=4, color="#444", label="data")
ax[2].plot(tt, model(x_lm, tt), "-", lw=2, color="#16a085", label="LM fit")
ax[2].plot(tt, model(x0_bad, tt), ":", lw=1.5, color="#e67e22", label="initial guess")
ax[2].set_xlabel("t"); ax[2].set_ylabel("y")
ax[2].set_title(r"Gaussian peak  $A\,e^{-(t-\mu)^2/2\sigma^2}$"); ax[2].legend(fontsize=8)

plt.tight_layout(); plt.show()
print("done")


done


## 4. 결과 해석

1. **LM 은 나쁜 초기값에서도 수렴**: 초기 $\lambda$ 가 커서 첫 스텝들은 최급강하에 가깝게 **짧고 안전**하게 움직인다. 봉우리 근처로 진입하면 $\rho\approx1$ 이 되어 $\lambda$ 가 줄고, 스텝이 **Gauss–Newton 으로 전환**되어 마지막에 빠르게 수렴한다(약 30회).
2. **순수 Gauss–Newton 은 폭주/진동**: 초기값이 봉우리에서 벗어나 $J^\top J$ 가 거의 특이하거나 스텝이 과도해, cost 가 줄지 않고 발산하거나 엉뚱한 점으로 튄다(감쇠가 없어 스텝 길이를 통제 못 함).
3. **최급강하는 수렴하나 매우 느림**: 안전하지만 골짜기에서 지그재그로 수백~수천 회 반복이 필요하다.
4. **$\lambda$ 궤적이 곧 적응 메커니즘**: 그래프 (b) 에서 $\lambda$ 가 초반엔 컸다가(=신중) 수렴 국면에서 급감(=공격적)하는 모습이 LM 의 "신뢰영역 자동 조절"을 보여준다. (참고: $\sigma$ 는 모형에 제곱으로만 들어가 부호가 임의이므로, 최종값은 절댓값으로 보고했다.)

> **결론**: Levenberg–Marquardt 는 $\lambda$ 하나로 **Gauss–Newton(빠름)과 최급강하(안전)** 를 매끄럽게 보간하여, 나쁜 초기값·특이 $J^\top J$ 에서도 강건하게 전역적으로 수렴한다.

**다음 문제로의 연결** — GN 과 LM 의 강건성 차이는 *초기값에 따라* 얼마나 벌어질까? 다음 노트북(§13.8-3)에서 **초기값 격자의 수렴 성공맵(basin of attraction)** 과 야코비안 **조건수**로 두 방법을 정량 비교한다.
